# Map Consumers to Lakehouse Layers — Solution


## Consumer-to-Layer Mapping

| Consumer | Layer | Access Method | Justification |
|----------|-------|---------------|---------------|
| Executive dashboards (Power BI) | **Gold** | Athena ODBC connector | Need pre-aggregated KPIs; low technical skill means they can't write SQL against raw data; gold provides governed, consistent metrics |
| Data science team | **Bronze** | Direct S3 read (PySpark/pandas) | Need raw transcripts for model training; schema-on-read flexibility; they handle their own transformations |
| Product managers | **Gold** | Athena ODBC (via Power BI or Athena console) | Customer health scores are a curated data product; pre-computed in gold; basic SQL sufficient |
| Real-time alerting service | **N/A — not from lakehouse** | Kinesis stream consumer (direct) | Seconds latency required; lakehouse is batch (min 4-hour refresh); must consume from the streaming layer directly |
| Finance team | **Gold** | Scheduled Athena query → S3 export → Excel/BI | Revenue reconciliation needs governed, auditable numbers; gold provides single source of truth; export to their preferred format |
| ML inference pipeline | **Silver** | Athena API (programmatic SQL) | Needs enriched customer profiles (cleansed, deduplicated) but not pre-aggregated; silver gives entity-level granularity at minutes-fresh |

## Alternative Serving Pattern

**Real-time alerting service** should NOT read from the lakehouse.

The lakehouse refreshes every 4 hours at best — escalation routing needs sub-second latency. The correct pattern is:

- Consume directly from the **Kinesis Data Stream** (same stream feeding the bronze layer)
- Use a Lambda function or Kinesis Data Analytics to filter for escalation events in real-time
- Route to SNS/PagerDuty immediately

The lakehouse serves as the system of record (bronze captures everything), but real-time consumers need the streaming layer, not the batch layer.

## Access Controls

**Data science team (Bronze access):**
- Lake Formation grants: READ on `bronze/clickstream/*` and `bronze/support_tickets/*`
- Column restriction: `customer_email` and `customer_phone` columns **masked** (PII) — data scientists get hashed values unless they request PII access with justification
- Row restriction: None (they need full dataset for training)

**Product managers (Gold access):**
- Lake Formation grants: READ on `gold/customer_health_scores` only
- Column restriction: `lifetime_revenue` column restricted to Finance role only — product managers see tier and health score but not revenue figures
- Row restriction: Product managers see only accounts in their assigned region (row-level filter on `region` column)